In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import messagebox
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input

In [74]:
data = pd.read_csv("pesticides.csv")
data.head()

,Domain,Area,Element,Item,Year,Unit,Value
0,Pesticides Use,Albania,Use,Pesticides (total),1990,tonnes of active ingredients,121.0
1,Pesticides Use,Albania,Use,Pesticides (total),1991,tonnes of active ingredients,121.0
2,Pesticides Use,Albania,Use,Pesticides (total),1992,tonnes of active ingredients,121.0
3,Pesticides Use,Albania,Use,Pesticides (total),1993,tonnes of active ingredients,121.0
4,Pesticides Use,Albania,Use,Pesticides (total),1994,tonnes of active ingredients,201.0


In [75]:
data.columns = data.columns.str.strip()
data.head()

,Domain,Area,Element,Item,Year,Unit,Value
0,Pesticides Use,Albania,Use,Pesticides (total),1990,tonnes of active ingredients,121.0
1,Pesticides Use,Albania,Use,Pesticides (total),1991,tonnes of active ingredients,121.0
2,Pesticides Use,Albania,Use,Pesticides (total),1992,tonnes of active ingredients,121.0
3,Pesticides Use,Albania,Use,Pesticides (total),1993,tonnes of active ingredients,121.0
4,Pesticides Use,Albania,Use,Pesticides (total),1994,tonnes of active ingredients,201.0


In [76]:
print(data["Area"].unique())

['Albania' 'Algeria' 'Angola' 'Antigua and Barbuda' 'Argentina' 'Armenia'
 'Australia' 'Austria' 'Azerbaijan' 'Bahamas' 'Bahrain' 'Bangladesh'
 'Barbados' 'Belarus' 'Belgium' 'Belgium-Luxembourg' 'Belize' 'Bermuda'
 'Bhutan' 'Bolivia (Plurinational State of)' 'Botswana' 'Brazil'
 'Brunei Darussalam' 'Bulgaria' 'Burkina Faso' 'Burundi' 'Cabo Verde'
 'Cameroon' 'Canada' 'Central African Republic' 'Chad' 'Chile'
 'China, Hong Kong SAR' 'China, Macao SAR' 'China, mainland'
 'China, Taiwan Province of' 'Colombia' 'Comoros' 'Congo' 'Cook Islands'
 'Costa Rica' "Côte d'Ivoire" 'Croatia' 'Cyprus' 'Czechia' 'Denmark'
 'Dominican Republic' 'Ecuador' 'Egypt' 'El Salvador' 'Eritrea' 'Estonia'
 'Ethiopia' 'Fiji' 'Finland' 'France' 'French Polynesia' 'Gambia'
 'Germany' 'Ghana' 'Greece' 'Guatemala' 'Guinea' 'Guinea-Bissau' 'Guyana'
 'Haiti' 'Honduras' 'Hungary' 'Iceland' 'India' 'Indonesia'
 'Iran (Islamic Republic of)' 'Iraq' 'Ireland' 'Israel' 'Italy' 'Jamaica'
 'Japan' 'Jordan' 'Kazakhstan' 'Keny

In [77]:
def train_model(country):
    df=data[data["Area"]==country]
    if df.empty:
        return None, None
    df=df.sort_values("Year")
    values=df["Value"].values.reshape(-1, 1)
    scaler=MinMaxScaler()
    values=scaler.fit_transform(values)
    X=[]
    y=[]
    for i in range(3, len(values)):
        X.append(values[i-3:i, 0])
        y.append(values[i, 0])
    X=np.array(X)
    y=np.array(y)
    X=X.reshape((X.shape[0], X.shape[1], 1))
    model=Sequential()
    model.add(Input(shape=(3, 1)))
    model.add(SimpleRNN(50,activation="tanh"))
    model.add(Dense(1))
    model.compile(optimizer="adam", loss="mse")
    model.fit(X, y, epochs=50, batch_size=4, verbose=0)
    return model, scaler

In [78]:
def predict_usage():
    country=country_entry.get()
    try:
        v1=float(year1_entry.get())
        v2=float(year2_entry.get())
        v3=float(year3_entry.get())
    except:
        messagebox.showerror("Please enter valid usage values")
        return
    model, scaler=train_model(country)
    if model is None:
        messagebox.showerror("Error,Country Not Found")
        return

    test=np.array([[v1], [v2], [v3]])
    test=scaler.transform(test)
    test=test.reshape((1, 3, 1))
    pred=model.predict(test, verbose=0)
    pred=scaler.inverse_transform(pred)
    result_label.config(text=f"Predicted Next Year Usage:{pred[0][0]:.2f} tonnes")

In [79]:
root=tk.Tk()
root.title("Pesticide Usage Prediction")
root.geometry("700x500")
root.config(bg="white")

In [80]:
title_label=tk.Label(root, text="Pesticide Usage Prediction", font=("Arial", 18, "bold"), bg="white")
title_label.pack(pady=15)

In [81]:
tk.Label(root, text="Enter Country Name", bg="white").pack()
country_entry = tk.Entry(root, width=30)
country_entry.pack(pady=5)
tk.Label(root, text="Enter Usage Year 1", bg="white").pack()
year1_entry = tk.Entry(root, width=30)
year1_entry.pack(pady=5)
tk.Label(root, text="Enter Usage Year 2", bg="white").pack()
year2_entry = tk.Entry(root, width=30)
year2_entry.pack(pady=5)
tk.Label(root, text="Enter Usage Year 3", bg="white").pack()
year3_entry = tk.Entry(root, width=30)
year3_entry.pack(pady=5)

In [82]:
predict_button = tk.Button(root,text="Predict",command=predict_usage,font=("Arial", 12, "bold"),bg="green",fg="white",width=15)
predict_button.pack(pady=20)

In [83]:
result_label = tk.Label(root,text="Prediction will appear here",font=("Arial", 12, "bold"),bg="white")
result_label.pack(pady=10)

In [84]:
root.mainloop()